In [7]:
import cv2
import numpy as np
import time
from collections import deque

# ─── تنظیمات ────────────────────────────────────────────
STOP_THRESHOLD  = 10000
STOP_DURATION   = 1.0
BLUR_KERNEL     = (7, 7)
INPUT_VIDEO     = 'fan.mp4'
OUTPUT_VIDEO    = 'fan_output.mp4'   # ← فایل خروجی
# ────────────────────────────────────────────────────────

cap     = cv2.VideoCapture(INPUT_VIDEO)
fps     = cap.get(cv2.CAP_PROP_FPS)
width   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# ─── تعریف VideoWriter ───────────────────────────────────
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

window_size     = int(fps * STOP_DURATION)
score_window    = deque(maxlen=window_size)
prev_gray       = None
alarm_triggered = False
stop_start_time = None
frame_idx       = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, BLUR_KERNEL, 0)

    motion_score = 0

    if prev_gray is not None:
        diff         = cv2.absdiff(prev_gray, gray)
        motion_score = np.sum(diff) / 255
        score_window.append(motion_score)

    prev_gray = gray

    # ─── تشخیص توقف ──────────────────────────────────────
    is_stopped = False

    if len(score_window) == window_size:
        avg_score  = np.mean(score_window)
        is_stopped = avg_score < STOP_THRESHOLD

        if is_stopped:
            if stop_start_time is None:
                stop_start_time = time.time()
            stopped_for = time.time() - stop_start_time
            if stopped_for >= STOP_DURATION and not alarm_triggered:
                alarm_triggered = True
        else:
            stop_start_time = None
            alarm_triggered = False
    else:
        avg_score = motion_score

    # ─── رسم UI ──────────────────────────────────────────
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (width, 60), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.7, frame, 0.3, 0, frame)

    # وضعیت
    if alarm_triggered:
        status       = "FAN STOPPED!"
        status_color = (0, 0, 255)
    elif is_stopped:
        stopped_for  = time.time() - (stop_start_time or time.time())
        remaining    = STOP_DURATION - stopped_for
        status       = f"Stopping... {remaining:.1f}s"
        status_color = (0, 165, 255)
    else:
        status       = "Fan Running"
        status_color = (0, 255, 80)

    cv2.putText(frame, status, (15, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.1, status_color, 2)

    # motion score
    cv2.putText(frame, f"Motion: {int(avg_score)}", (width - 200, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 1)

    # نوار motion
    bar_max  = 30000
    bar_w    = width - 20
    bar_fill = int(min(avg_score / bar_max, 1.0) * bar_w)
    bar_color = (0, 255, 80) if not is_stopped else (0, 0, 255)
    cv2.rectangle(frame, (10, 65), (10 + bar_w, 75), (50, 50, 50), -1)
    cv2.rectangle(frame, (10, 65), (10 + bar_fill, 75), bar_color, -1)

    # خط threshold
    thresh_x = int((STOP_THRESHOLD / bar_max) * bar_w) + 10
    cv2.line(frame, (thresh_x, 62), (thresh_x, 78), (0, 200, 255), 2)
    cv2.putText(frame, "threshold", (thresh_x - 30, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 200, 255), 1)

    # پیشرفت پردازش
    progress = int((frame_idx / total) * (width - 20))
    cv2.rectangle(frame, (10, height - 12), (10 + width - 20, height - 6), (50, 50, 50), -1)
    cv2.rectangle(frame, (10, height - 12), (10 + progress, height - 6), (100, 200, 255), -1)
    cv2.putText(frame, f"{int(frame_idx/total*100)}%", (10, height - 18),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (150, 150, 150), 1)

    # ─── ذخیره فریم ──────────────────────────────────────
    out.write(frame)

    cv2.imshow('Fan Stop Detection', frame)
    if cv2.waitKey(5) == 27:
        break

cap.release()
out.release()
cv2.destroyAllWindows()

print(f"ویدیو ذخیره شد: {OUTPUT_VIDEO}")

ویدیو ذخیره شد: fan_output.mp4
